# 01 — Supervised Fine-Tuning (SFT)

This notebook fine-tunes **Qwen/Qwen2.5-7B-Instruct** on the GSM8K dataset
using QLoRA + LoRA adapters via the `math_rl_tuning` package.

**Requirements:** Google Colab with GPU (T4 minimum, A100 recommended).

## 1. Setup — Install & Clone

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/math-rl-tuning.git
%cd math-rl-tuning

# Install the package and all dependencies
!pip install -e . --quiet
!pip install bitsandbytes --quiet

## 2. Configuration

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

# Load default config (edit configs/default.yaml to customize)
cfg = load_config()

# --- Authentication ---
# Option A: Set your tokens here
# setup_hf_token("hf_YOUR_TOKEN")
# setup_wandb(cfg.sft_training.wandb_project, key="YOUR_WANDB_KEY")

# Option B: Use Colab secrets (recommended)
setup_hf_token()  # reads from Colab secrets
setup_wandb(cfg.sft_training.wandb_project)

# Mount Google Drive for saving
mount_google_drive()

## 3. (Optional) Customize Config

You can override any config value programmatically:

In [ ]:
# Example: train for 2 epochs with a larger batch size
# cfg.sft_training.num_train_epochs = 2
# cfg.sft_training.per_device_train_batch_size = 8

# Example: use different data sources
# cfg.dataset.sft_keep_sources = ["gsm8k", "math", "cn_k12"]

# Example: change LoRA rank
# cfg.lora.sft.r = 32
# cfg.lora.sft.alpha = 64

## 4. Prepare Data

In [ ]:
from math_rl_tuning.data import prepare_sft_data

train_ds, val_ds = prepare_sft_data(cfg)

print(f"\nTrain examples: {len(train_ds)}")
print(f"Val examples:   {len(val_ds)}")
print(f"\nSample (first message):")
print(train_ds[0]["messages"][0]["content"][:500])

## 5. Run SFT Training

In [ ]:
from math_rl_tuning.sft_trainer import run_sft_training

trainer, model, tokenizer = run_sft_training(
    cfg,
    train_ds=train_ds,
    val_ds=val_ds,
    save_to_drive=True,  # auto-copies to Google Drive
)

## 6. Quick Sanity Check

In [ ]:
from math_rl_tuning.inference import generate_stream

question = "Solve x + y = 10, 2x - y = 30."
print(f"Question: {question}\n")
response = generate_stream(question, model, tokenizer)

## 7. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()